# 04 · Risultati

Tabelle e grafici a partire da tutti i file `punteggi_*.csv` in `artifacts/esperimenti/`.
Le tabelle vengono salvate anche in `artifacts/risultati/` per copiarle nella tesi.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path("..").resolve()))   # per importare la cartella tesi/ del progetto

import pandas as pd
import matplotlib.pyplot as plt

from tesi import percorsi, risultati, training

cartella_risultati = percorsi.PROGETTO / "artifacts" / "risultati"
cartella_risultati.mkdir(parents=True, exist_ok=True)

## 1. Esperimenti fatti

In [ ]:
esperimenti = risultati.leggi_esperimenti()
esperimenti.drop(columns="cartella")

## 2. Metriche per seed e media sui seed

EER in percentuale; minDCF, actDCF e Cllr come in ASVspoof 5. `EER std` è la deviazione standard sui seed.

In [ ]:
per_seed = risultati.tabella_seed()
media = risultati.tabella_media(per_seed)
per_seed.to_csv(cartella_risultati / "metriche_per_seed.csv", index=False)
media.to_csv(cartella_risultati / "metriche_media_seed.csv", index=False)
media

## 3. Tabella train x test

Righe: modello, dataset di train e augmentation. Colonne: liste di test. Valori: EER medio sui seed (%).

In [ ]:
eer = risultati.matrice(media, "EER (%)")
eer.to_csv(cartella_risultati / "matrice_eer.csv")
eer

In [ ]:
risultati.matrice(media, "minDCF")

## 4. Confronto delle augmentation

Per ogni modello e dataset di train, EER medio con ogni augmentation, test per test.

In [ ]:
confronto = media.pivot_table(index=["modello", "train", "test"], columns="augmentation", values="EER (%)").round(2)
confronto

In [ ]:
MODELLO, TRAIN = "mhfa", "asvspoof5"

dati_grafico = confronto.loc[(MODELLO, TRAIN)]
ax = dati_grafico.plot.bar(figsize=(9, 4), rot=0)
ax.set(ylabel="EER (%)", xlabel="", title=f"{MODELLO}, allenato su {TRAIN}: EER per augmentation")
plt.tight_layout()
plt.savefig(cartella_risultati / f"augmentation_{MODELLO}_{TRAIN}.png", dpi=200)

## 5. EER per attacco e per codec

Per un esperimento e un seed: gli spoof di ogni attacco (o codec) contro tutti i bonafide.

In [ ]:
cartella = training.cartella_esperimento("mhfa", "asvspoof5", "rawboost", 1)
pd.DataFrame({
    "EER per attacco (%)": risultati.eer_attacchi(cartella, "asvspoof5_eval"),
}).round(2)

In [ ]:
risultati.eer_attacchi(cartella, "asvspoof5_eval", colonna="codec").round(2)